# 00 — Bronze: bidsIngests the raw bid export exactly as it arrives. **No casting, no cleaning,no filtering.** Anything that looks wrong here is preserved so the Silverlayer can decide what to do about it, and so the raw layer stays a faithfulcopy of the source.Schema drift is accepted rather than rejected: the source is a manual Excelexport whose columns change without notice, and failing the load on acosmetic change would stop the pipeline for no good reason. The datacontract is enforced at Silver.

In [0]:
CATALOG = "bronze"SCHEMA = "bid"VOLUME_PATH = "/Volumes/raw/bid/bids/bids.xlsx"TABLE = f"{CATALOG}.{SCHEMA}.bids"spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

## Read`header=true` promotes the first row. The `dataAddress` names the sheet only —never a fixed cell range. A hardcoded range such as `Bronze!A1:K1583` silentlytruncates the moment the source grows by one row, which is the worst class ofbug: no error, just missing data.

In [0]:
df_raw = (    spark.read.format("com.crealytics.spark.excel")    .option("header", "true")    .option("inferSchema", "false")   # everything lands as string; Silver casts    .option("dataAddress", "'Bronze'!A1")    .load(VOLUME_PATH))print(f"rows: {df_raw.count()}   columns: {len(df_raw.columns)}")df_raw.printSchema()

## Ingestion metadata`_ingested_at` and `_source_file` make it possible to answer "when did this rowarrive and where did it come from" months later, without which incidenttriage on a data pipeline is guesswork.

In [0]:
from pyspark.sql import functions as Fdf_bronze = (    df_raw    .withColumn("_ingested_at", F.current_timestamp())    .withColumn("_source_file", F.lit(VOLUME_PATH)))(    df_bronze.write    .format("delta")    .mode("overwrite")    .option("mergeSchema", "true")   # deliberate: see note below    .saveAsTable(TABLE))

### On `mergeSchema`Accepting schema evolution at Bronze is a choice, not a default. The upstreamexport gains and loses columns without warning; refusing them would breakingestion for a change that costs nothing to absorb. The cost is that arenamed column arrives silently as a new one — which is why the check belowexists and why Silver validates against an explicit contract.

In [0]:
EXPECTED = {    "bid_id", "created_at", "created_at_str", "is_confirmed_date", "bid_date",    "closed_at", "closed_at_str", "outcome", "loss_reason", "competitor_name",    "client_id", "contract_value_brl",}actual = set(df_bronze.columns) - {"_ingested_at", "_source_file"}missing, unexpected = EXPECTED - actual, actual - EXPECTEDif missing:    raise ValueError(f"Columns missing from source: {sorted(missing)}")if unexpected:    print(f"WARNING — new columns absorbed, review Silver: {sorted(unexpected)}")print(f"Schema check passed. {spark.table(TABLE).count()} rows written to {TABLE}.")